# Surrogate pipeline — feature store to submission CSV

One notebook for every case of the challenge, from the Hopsworks feature store to the CSV the
organisers score. Run it top to bottom; every choice sits in a `CONFIG` cell at the top of its step.

| Step | What happens | Stored in Hopsworks |
|---|---|---|
| 1 | Look at the feature groups | – |
| 2 | Pick the case (trajectory, direct B, inverse B, inverse A) and its input features | – |
| 3 | Choose the input normalisations; build the feature view and the train / holdout split | feature view, transformation functions, training-dataset statistics |
| 4 | Train any model defined in a `.py` file under `models/` | – |
| 5 | Score the holdout with `pallas_score.py`, against the objective's reference | – |
| 6 | Upload the model, its code and pinned dependencies to the model registry | model |
| 7 | Benchmark every model registered for the case, against the objective's reference | – |
| 8 | Load a model back from the registry, predict the test file, write the submission CSV | – |

**The hand-in is the CSV.** Upload it to DALIA; the organisers score it on the hidden test and
publish the score on W&B. The holdout numbers in steps 5 and 7 are our own guide, not the
official score.

**Before the first run** (once per machine, from this folder):

```bash
uv venv --python 3.11 .venv
uv pip install --python .venv/bin/python -r requirements.txt
cp .env.example ../../.env        # then fill in HOPSWORKS_API_KEY and WANDB_API_KEY
.venv/bin/python check_setup.py   # must end with "all good"
```

Then pick the `.venv` interpreter as this notebook's kernel. Data is read from the Hopsworks
team project `m4cast_plasma_pirates`, which holds the full public pack; the test inputs are read from
`data/pallas_hackathon_data/` (not in git: copy the pack there).

**Naming:** objects you create carry your prefix (`OWNER`), e.g. `grl_trajectory_fv`,
`grl_trajectory_linear_baseline`, so teammates' versions never collide.

In [ ]:
import dataclasses
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import pallas_pipeline as pp
import pallas_score

OWNER = "grl"                      # your prefix for everything this notebook creates
project = pp.connect(pp.PROJECT)   # m4cast_plasma_pirates: the full public pack
fs = project.get_feature_store()

## 1. The feature store

The four feature groups hold the whole public pack, one row per simulation (per position for
the trajectories, per plane for the moments). Column names are the pack's names in lower case.

In [ ]:
pp.feature_groups_overview(fs)

In [ ]:
FEATURE_GROUP = "pallas_b_trajectories"    # any of the names above
display(pp.feature_group_schema(fs, FEATURE_GROUP).T)
fs.get_feature_group(FEATURE_GROUP, version=1).show(5)

## 2. The case to solve

`CASES` follows `objectives.md`: inputs, targets, the rows the objective trains on, the test file
and the reference number. `INPUTS` is the feature set the model sees; you can drop inputs, but not add ones the
test file does not carry (the CSV has to be predicted from that file alone).

In [ ]:
# ---- CONFIG -------------------------------------------------------------------------
CASE = "trajectory"                # trajectory | direct_B | inverse_B | inverse_A
INPUTS = None                      # None = the objective's inputs; or a sublist of them
# -------------------------------------------------------------------------------------
case = pp.CASES[CASE]
if INPUTS is not None:
    case = dataclasses.replace(case, inputs=list(INPUTS))
test_columns = pd.read_parquet(pp.DATA_DIR / case.test_file).columns
missing = [c for c in case.inputs if c not in test_columns]
assert not missing, f"{missing} are not in {case.test_file}: the submission could not be predicted"

pd.Series({"feature group": case.feature_group, "inputs": case.inputs, "targets": case.targets,
           "test file": case.test_file, "submission": case.submission_file,
           "reference (objectives.md)": pp.reference_scores(case),
           "what the reference is": case.reference_note})

## 3. Normalisations and the data pipeline

**Input normalisations** are stored in the feature view as Hopsworks transformation functions —
`standard`, `min_max`, `robust`, `log10` (our `grl_log10`, saved in the feature store on first
use) or `none` — so whoever reads the view gets the same inputs. Their statistics come from a
training dataset computed once over the view's rows (inputs only, so the holdout's targets do not
leak).

**Targets** are scaled inside the model (`TARGET_SCALING` in step 4), because predictions have to
go back to physical units and Hopsworks does not invert a transformation.

A feature view version is immutable: to change the normalisations or the inputs, raise
`FV_VERSION`. The rows (e.g. injected bunches only) come from the case.

In [ ]:
# ---- CONFIG -------------------------------------------------------------------------
NORMALISATION = case.default_normalisation()   # {input: standard|min_max|robust|log10|none}
# NORMALISATION["x_of"] = "min_max"            # edit any entry
FV_NAME = f"{OWNER}_{CASE}_fv".lower()      # Hopsworks names are lower case
FV_VERSION = 1
# -------------------------------------------------------------------------------------
fv = pp.get_feature_view(fs, FV_NAME, FV_VERSION)
if fv is None:
    fv = pp.create_feature_view(fs, case, FV_NAME, FV_VERSION, NORMALISATION,
                                description=f"{CASE}: {', '.join(case.inputs)} -> "
                                            f"{', '.join(case.targets)}")
    print(f"created {FV_NAME} v{FV_VERSION}")
else:
    print(f"using the existing {FV_NAME} v{FV_VERSION}")

stored = pp.feature_view_normalisation(fv)
wanted = {pp.fs_name(c): k for c, k in NORMALISATION.items() if k != "none"}
pd.DataFrame({"stored in the view": pd.Series(stored), "asked for here": pd.Series(wanted)})

If the two columns above disagree, the view was created with other settings: raise `FV_VERSION`
to store yours. Next: the statistics, the rows, and the split. The split is always on whole
configurations — rows of one simulation never sit on both sides.

- `random`: a random `HOLDOUT_FRACTION` of the configurations.
- `ood_density` (trajectory only): the **Hard out-of-distribution objective**. The model trains
  only inside `ood_split.train` (P_max < 6,528.6 Pa) and is scored on two groups: `in_range`, a
  random `HOLDOUT_FRACTION` of those configurations, and `above_cut`, the 578 public
  configurations above the cut. The deliverable is the gap between the two. The model is
  registered with an `_ood` suffix and step 8 writes `submission_ood.csv`.

Note the hidden trajectory test leans to high density (38 % above the cut), so a random holdout
reads optimistic.

In [ ]:
# ---- CONFIG -------------------------------------------------------------------------
SPLIT = "random"                   # random | ood_density
HOLDOUT_FRACTION = 0.2
SEED = 0
# -------------------------------------------------------------------------------------
TD_VERSION = pp.prepare_statistics(fv)
rows = pp.read_rows(fv, case)
train_configs, holdout_configs, holdout_groups = pp.split_configs(rows, case, SPLIT,
                                                                  HOLDOUT_FRACTION, SEED)

train_rows = rows[rows.config.isin(train_configs)].reset_index(drop=True)
holdout_rows = rows[rows.config.isin(holdout_configs)].reset_index(drop=True)
X_train = pp.transform(fv, train_rows, case)
X_holdout = pp.transform(fv, holdout_rows, case)
FEATURE_COLUMNS = list(X_train.columns)

print(f"training-dataset statistics: v{TD_VERSION}")
print(f"train   {len(train_configs):5d} configurations, {len(train_rows):7,d} rows")
for group, members in holdout_groups.items():
    n_rows = int(rows.config.isin(members).sum())
    print(f"{group:9s} {len(members):5d} configurations, {n_rows:7,d} rows (held out)")
X_train.describe().T[["mean", "std", "min", "max"]]

## 4. Train a model from a `.py` file

A model file defines `build_model(**params)`, returning an object with `fit(X, Y)` and
`predict(X)` on numpy arrays (`Y` has one column per target), and `PARAMS`, its defaults. Three
examples live in `models/`: `linear_baseline.py` (the bar the Easy objectives ask you to beat),
`mlp_sklearn.py` and `torch_mlp.py` (CUDA or Apple MPS when available). Copy one to start your own.

`TARGET_SCALING = "scorer"` trains on the scale R² is computed on (log10 for charge, spread,
emittance, ...); predictions come back in physical units either way.

In [ ]:
# ---- CONFIG -------------------------------------------------------------------------
MODEL_FILE = "models/linear_baseline.py"
PARAMS = {}                        # overrides of the file's PARAMS
TARGET_SCALING = "scorer"          # scorer | none
# -------------------------------------------------------------------------------------
model_module = pp.load_model_file(MODEL_FILE)
params = {**getattr(model_module, "PARAMS", {}), **PARAMS}
print(f"{Path(MODEL_FILE).stem} with {params}")

started = time.time()
model = pp.SurrogateModel(model_module.build_model(**params), case.targets, TARGET_SCALING)
model.fit(X_train, train_rows)
TRAIN_SECONDS = time.time() - started
print(f"trained in {TRAIN_SECONDS:.1f} s")

## 5. Results on the holdout

Scored with `pallas_score.py`, the organisers' own scorer, on configurations the model never saw.
For the trajectory the ranked number is the in-plasma one, R² per position against the per-z mean;
the drift is shown for information. With `SPLIT = "ood_density"` each holdout group is scored on
its own and the table shows the gap; `scores` is then the above-cut one.

In [ ]:
holdout_pred = pp.predictions(case, model, X_holdout, holdout_rows)
holdout_key = pp.answer_key(case, holdout_rows)
in_group = lambda frame, members: frame[frame.config.isin(members)].reset_index(drop=True)
group_scores = {group: pallas_score.score_submission(in_group(holdout_pred, members),
                                                     in_group(holdout_key, members))
                for group, members in holdout_groups.items()}
scores = group_scores.get("above_cut", group_scores.get("holdout"))   # what this split is about

ref = pp.reference_scores(case)
rows_shown = case.targets + ["mean"]
summary = pd.DataFrame({**{f"{group} R2": {t: s[t] for t in rows_shown}
                           for group, s in group_scores.items()},
                        "reference": {t: ref.get(t) for t in rows_shown}})
if "above_cut" in group_scores:
    summary["gap (in range - above cut)"] = summary["in_range R2"] - summary["above_cut R2"]
    print(f"mean R2 in range {group_scores['in_range']['mean']:.4f}, above the cut "
          f"{scores['mean']:.4f}: gap {group_scores['in_range']['mean'] - scores['mean']:.4f}")
else:
    summary["above the reference"] = summary["holdout R2"] > summary["reference"]
    print(f"mean R2 on the holdout: {scores['mean']:.4f}  (reference {ref['mean']:.4f}: "
          f"{case.reference_note})")
summary

In [ ]:
if case.positional:
    display(pallas_score.score_zones(holdout_pred, holdout_key))

In [ ]:
# predicted vs true, on the scale R2 is computed on
merged = holdout_key.merge(holdout_pred, on=case.ids, suffixes=("", "_pred"))
shown = merged.sample(min(len(merged), 20_000), random_state=0)
fig, axes = plt.subplots(1, len(case.targets), figsize=(3.2 * len(case.targets), 3.2))
for ax, t in zip(np.atleast_1d(axes), case.targets):
    for group, members in holdout_groups.items():
        part = shown[shown.config.isin(members)]
        true = pallas_score.physical_to_normalised(t, part[t])
        pred = pallas_score.physical_to_normalised(t, part[f"{t}_pred"])
        ax.scatter(true, pred, s=2, alpha=0.3, label=group)
    edges = ax.get_xlim()
    ax.plot(edges, edges, "k--", lw=1)
    title = " / ".join(f"{s[t]:.3f}" for s in group_scores.values())
    ax.set(title=f"{t}  R2 {title}", xlabel="true", ylabel="predicted")
if len(holdout_groups) > 1:
    np.atleast_1d(axes)[0].legend(markerscale=5, fontsize=8)
fig.tight_layout()

In [ ]:
# trajectory: where along the accelerator the model loses the configurations
if case.positional:
    by_z = pp.r2_by_position(case, holdout_pred, holdout_key)
    ax = by_z.plot(x="z_mm", y=case.targets, figsize=(9, 4), ylim=(-0.5, 1.02))
    ax.axvspan(3.6, 4.0, color="grey", alpha=0.2, label="plasma end (L_inj + 3.2 mm)")
    ax.set(ylabel="R2 across configurations at this z", title="R2 per position, holdout")
    ax.legend(ncol=4, fontsize=8)

Optional: log this experiment to the team's W&B group, to compare models across teammates.
It writes `holdout/...` keys only; the official `pallas/mean_r2` is published by the organisers.

In [ ]:
LOG_TO_WANDB = False
if LOG_TO_WANDB:
    from team_wandb import log_scores, start_run
    run = start_run(task=CASE, tier="easy", name=f"{OWNER}-{CASE}-{Path(MODEL_FILE).stem}",
                    tags=[OWNER, CASE], model_file=Path(MODEL_FILE).stem, params=params,
                    target_scaling=TARGET_SCALING, normalisation=NORMALISATION,
                    feature_view=f"{FV_NAME}:v{FV_VERSION}", split=SPLIT, seed=SEED,
                    train_seconds=TRAIN_SECONDS)
    log_scores(run, scores)
    run.finish()

## 6. Upload the model to the model registry

The registry entry is self-contained: the fitted model (`model.joblib`), the model file and the
code needed to unpickle it (`pallas_pipeline.py`, `pallas_score.py`), `requirements.txt` with the
versions installed here, and `model_meta.json` (case, inputs, normalisations, split, holdout
configurations). It is linked to the feature view and to the training dataset whose statistics
the normalisations use, so inference reproduces the exact inputs. Each upload is a new version.

In [ ]:
MODEL_NAME = f"{OWNER}_{CASE}_{Path(MODEL_FILE).stem}".lower()
if SPLIT == "ood_density":
    MODEL_NAME += "_ood"           # the Hard objective's model: kept apart from the others
DESCRIPTION = (f"{CASE}: {Path(MODEL_FILE).stem} {params}, targets on the {TARGET_SCALING} "
               f"scale, {FV_NAME} v{FV_VERSION}, {SPLIT} holdout (seed {SEED})")

meta = {"case": CASE, "inputs": case.inputs, "targets": case.targets,
        "feature_columns": FEATURE_COLUMNS, "normalisation": NORMALISATION,
        "feature_view": FV_NAME, "feature_view_version": FV_VERSION, "td_version": TD_VERSION,
        "model_file": Path(MODEL_FILE).stem, "params": params, "target_scaling": TARGET_SCALING,
        "split": SPLIT, "seed": SEED, "holdout_configs": holdout_configs,
        "holdout_groups": holdout_groups,
        "holdout_scores": {group: {k: v for k, v in s.items() if isinstance(v, float)}
                           for group, s in group_scores.items()}}
metrics = {"holdout_mean_r2": scores["mean"],
           **{f"holdout_r2_{pp.fs_name(t)}": scores[t] for t in case.targets}}
if "in_range" in group_scores:     # OOD: holdout_mean_r2 is the above-cut score
    metrics["holdout_in_range_mean_r2"] = group_scores["in_range"]["mean"]
    metrics["holdout_gap"] = group_scores["in_range"]["mean"] - scores["mean"]

bundle = pp.save_bundle(model, MODEL_FILE, meta, Path("outputs/models") / MODEL_NAME)
registered = pp.register_model(project, bundle, MODEL_NAME, fv, TD_VERSION, metrics, DESCRIPTION)
MODEL_VERSION = registered.version
print(f"registered {MODEL_NAME} v{MODEL_VERSION}: {sorted(p.name for p in bundle.iterdir())}")

## 7. Benchmark of the models registered for this case

Every model in the registry whose name carries the case, best first, with the scores it was
registered with and the objective's reference on top. Read it as a guide, not a ranking:

- `from_notebook` models were scored on their own holdout (the split is in `description`); two
  models compare fairly only when they share the split and seed.
- `_ood` models (Hard objective) report their above-cut score; the in-range score and the gap
  are in their registry metrics.
- Other models (e.g. `pallas_direct_b`) report whatever their author measured, on their own
  split, and cannot be loaded in step 8.
- The official number is the organisers' score of the CSV on the hidden test.

In [ ]:
benchmark = pp.model_benchmark(project, CASE)
ref = pp.reference_scores(case)
ref_row = pd.DataFrame([{"model": "reference (objectives.md)", "mean_r2": ref["mean"],
                         **{t: ref.get(t) for t in case.targets},
                         "description": case.reference_note}])
table = pd.concat([ref_row, benchmark], ignore_index=True)
if "version" in table:
    table["version"] = table["version"].astype("Int64")
table.style.format(
    {c: "{:.4f}" for c in ["mean_r2"] + case.targets}, na_rep="")

In [ ]:
if not benchmark.empty:
    shown = benchmark.dropna(subset=["mean_r2"]).iloc[::-1]
    labels = shown["model"] + " v" + shown["version"].astype(str)
    colours = ["tab:blue" if ours else "tab:grey" for ours in shown["from_notebook"]]
    fig, ax = plt.subplots(figsize=(8, 0.4 * len(shown) + 1.2))
    ax.barh(labels, shown["mean_r2"], color=colours)
    ax.axvline(ref["mean"], color="tab:red", ls="--", label=f"reference {ref['mean']:.3f}")
    ax.set(xlabel="mean R2 as registered (blue: this notebook, own holdout)",
           xlim=(min(0, shown["mean_r2"].min()), 1), title=f"{CASE}: registered models")
    ax.legend(loc="lower right")
    fig.tight_layout()

## 8. Inference on the test file and the submission CSV

Loads a model back **from the registry** — any teammate can run this cell alone with a name
and version from the benchmark above (a `from_notebook` one) — applies the feature view's
normalisations to the test inputs, and writes the CSV in the scorer's format (id columns, then
one column per target, physical units; positions with 6 decimals, the rest at full precision).

In [ ]:
# ---- CONFIG -------------------------------------------------------------------------
INFER_NAME, INFER_VERSION = MODEL_NAME, MODEL_VERSION   # or any from_notebook row above
# -------------------------------------------------------------------------------------
loaded, loaded_meta = pp.load_registered_model(project, INFER_NAME, INFER_VERSION,
                                               "outputs/downloaded")
infer_case = dataclasses.replace(pp.CASES[loaded_meta["case"]], inputs=loaded_meta["inputs"])
infer_fv = fs.get_feature_view(loaded_meta["feature_view"], loaded_meta["feature_view_version"])
if infer_fv.transformation_functions:
    infer_fv.init_batch_scoring(training_dataset_version=loaded_meta["td_version"])

test = pd.read_parquet(pp.DATA_DIR / infer_case.test_file)
X_test = pp.transform(infer_fv, test, infer_case)[loaded_meta["feature_columns"]]
submission = pp.predictions(infer_case, loaded, X_test, test)

SUBMISSION_FILE = pp.submission_file(infer_case, loaded_meta.get("split"))   # OOD: submission_ood.csv
out = Path("submissions") / f"{INFER_NAME}_v{INFER_VERSION}" / SUBMISSION_FILE
pp.write_submission(infer_case, submission, test, out)
print(f"wrote {out} ({len(submission):,} rows)")
submission.head()

**Upload to DALIA.** Copy the CSV to the submission folder on DALIA, from a terminal on this
machine (`DALIA_DIR` is relative to your DALIA home):

In [ ]:
DALIA_DIR = "Downloads"            # TODO: the organisers' submission folder, once confirmed
print(f"scp '{out.resolve()}' dalia:{DALIA_DIR}/{SUBMISSION_FILE}")